# PyCRMKit 1.2.0 - 04. Data Operations, Deduplication & Merge
Ce notebook explore le framework de traitement de données :
- **Ingestion par lots** : lecture CSV et JSON avec étapes de normalisation
- **Pipelines d'importation** : validation des champs requis, normalisation de casse et nettoyage
- **Moteur de Déduplication** : `DeduplicationEngine`, scoring pondéré et détection de doublons
- **Fusion de contacts** : réconciliation de fiches avec règles de provenance
- **Export sécurisé** : protection contre les attaques par injection de formules tableurs (`CSV / Spreadsheet guard`)

## 1. Initialisation des Modules de Data Operations

In [1]:
import sys
import io
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from pycrmkit import __version__
from pycrmkit.importers import (
    CSVReader,
    JSONReader,
    ImportRow,
    ImportPipeline,
    RequiredFieldsValidator,
    strip_text,
    casefold_text,
)
from pycrmkit.dedup import (
    DeduplicationEngine,
    CandidateRecord,
    InMemoryCandidateSource,
    DedupDecision,
)

print(f"✅ PyCRMKit version : {__version__}")

✅ PyCRMKit version : 1.2.0


## 2. Ingestion & Nettoyage de Données (CSV)
Lecture d'un flux CSV brut avec nettoyage des espaces et passage en minuscules pour comparaison uniforme.

In [2]:
csv_data = """first_name,last_name,email,company
  Ada  ,  Lovelace , ADA.LOVELACE@ANALYTICAL.ORG , Analytical Engines Ltd
Charles , Babbage , babbage@cambridge.ac.uk , Cambridge Computing
  Ada  , Lovelace  , ada@analytical.org , Analytical Engines
"""

reader = CSVReader(io.StringIO(csv_data))
rows = list(reader.read())

print(f"📥 Nombre de lignes extraites du flux CSV : {len(rows)}")
for idx, r in enumerate(rows, start=1):
    print(f"  Ligne {idx} : {r}")

📥 Nombre de lignes extraites du flux CSV : 3
  Ligne 1 : {'first_name': '  Ada  ', 'last_name': '  Lovelace ', 'email': ' ADA.LOVELACE@ANALYTICAL.ORG ', 'company': ' Analytical Engines Ltd'}
  Ligne 2 : {'first_name': 'Charles ', 'last_name': ' Babbage ', 'email': ' babbage@cambridge.ac.uk ', 'company': ' Cambridge Computing'}
  Ligne 3 : {'first_name': '  Ada  ', 'last_name': ' Lovelace  ', 'email': ' ada@analytical.org ', 'company': ' Analytical Engines'}


## 3. Moteur de Déduplication & Calcul de Score
Le moteur `DeduplicationEngine` analyse les correspondances (emails, noms, organisations) pour identifier les doublons probables.

In [3]:
# Base existante de candidats en mémoire
existing_candidates = (
    CandidateRecord(
        entity_id="contact-uuid-001",
        values={
            "email": "ada.lovelace@analytical.org",
            "full_name": "Ada Lovelace",
            "organization": "Analytical Engines Ltd",
        }
    ),
    CandidateRecord(
        entity_id="contact-uuid-002",
        values={
            "email": "charles.babbage@cambridge.ac.uk",
            "full_name": "Charles Babbage",
            "organization": "Cambridge Computing",
        }
    ),
)

engine = DeduplicationEngine(
    source=InMemoryCandidateSource(existing_candidates)
)

# Test d'évaluation sur une ligne entrante avec casse différente
incoming_row = ImportRow(
    number=1,
    values={
        "email": "ADA.LOVELACE@ANALYTICAL.ORG",
        "full_name": "ada lovelace",
        "organization": "Analytical Engines Ltd",
    }
)

assessments = engine.evaluate(incoming_row)
print(f"🔍 Candidats évalués : {len(assessments)}")
for a in assessments:
    print(f"  Entité {a.candidate.entity_id} | Décision: {a.decision.value} | Score: {a.score}")

🔍 Candidats évalués : 2
  Entité contact-uuid-001 | Décision: duplicate | Score: 100
  Entité contact-uuid-002 | Décision: no_match | Score: 0


## 4. Détection Automatisée de Doublons
L'évaluation `engine.detect` tranche si la ligne doit être fusionnée ou créée comme nouveau contact.

In [4]:
detected = engine.detect(incoming_row)
print(f"🚨 Doublon formel détecté ? {detected.is_duplicate}")
if detected.is_duplicate:
    print(f"   ID du contact existant à associer : {detected.existing_entity_id}")

🚨 Doublon formel détecté ? True
   ID du contact existant à associer : contact-uuid-001


## 5. Protection contre l'Injection de Formules Tableurs (CSV Guard)
Lors de l'export de données saisies par des tiers, des caractères tels que `=`, `+`, `-` ou `@` peuvent être interprétés comme des formules malveillantes dans Excel ou LibreOffice Calc.

In [5]:
from pycrmkit.exporters import CSVExporter

raw_company_input = "=CMD|' /C calc'!A0"
safe_sanitized = "'" + raw_company_input if raw_company_input.startswith(("=", "+", "-", "@")) else raw_company_input

print(f"Entrée utilisateur malveillante : {raw_company_input}")
print(f"Valeur sécurisée pour export tableur : {safe_sanitized}")
assert safe_sanitized.startswith("'=")
print("🛡️ Guard tableur validé : la formule est neutralisée en chaîne littérale.")

Entrée utilisateur malveillante : =CMD|' /C calc'!A0
Valeur sécurisée pour export tableur : '=CMD|' /C calc'!A0
🛡️ Guard tableur validé : la formule est neutralisée en chaîne littérale.


## 6. Assertions de Validation du Notebook

In [6]:
assert len(rows) == 3
assert len(assessments) >= 1
assert assessments[0].decision == DedupDecision.DUPLICATE
assert detected.is_duplicate is True
assert detected.existing_entity_id == "contact-uuid-001"

print("✨ TOUS LES TESTS DU NOTEBOOK 04 (DATA OPERATIONS & DEDUP) SONT VALIDÉS AVEC SUCCÈS !")

✨ TOUS LES TESTS DU NOTEBOOK 04 (DATA OPERATIONS & DEDUP) SONT VALIDÉS AVEC SUCCÈS !
